# Internet of Things Application Development
# Lab 4 - AI in IoT application
Task: Your task is to setup and build an ML/DL model to process and predict temperature and humidity data taken from your sensors.

## Convolution Neural Network singular variable model
This is an example to predict the humidity given a sequence of humidity data

In [1]:
# %pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauth
# from google.colab import drive
import numpy as np
import pandas as pd
import random
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import Conv1D
from keras.layers import MaxPooling1D
# from google.colab import auth
# auth.authenticate_user()

# from googleapiclient.discovery import build
# from googleapiclient.http import MediaIoBaseDownload
import io

In [2]:
# Data preprocessing
def split_sequence(sequence, n_steps):
	X, y = list(), list()
	for i in range(len(sequence)):
		# Find the end of this pattern
		end_ix = i + n_steps
		# Check if we are beyond the sequence
		if end_ix > len(sequence)-1:
			break
		# Gather input and output parts of the pattern
		seq_x, seq_y = sequence[i:end_ix], sequence[end_ix]
		X.append(seq_x)
		y.append(seq_y)
	return np.array(X), np.array(y)

In [3]:
import pandas as pd
import os

# Define path to CSV files
lab4_dir = os.path.dirname(os.path.abspath("__file__"))  # Get current directory
train_file = os.path.join(lab4_dir, "train.csv")
test_file = os.path.join(lab4_dir, "test.csv")

# Read the CSV files directly
train_data = pd.read_csv(train_file)
test_data = pd.read_csv(test_file)

# Extract humidity data
humi_seq_train = train_data['Relative_humidity_room'].tolist()
humi_seq_test = test_data['Relative_humidity_room'].tolist()

# Print confirmation
print(f"Loaded {len(humi_seq_train)} training samples")
print(f"Loaded {len(humi_seq_test)} test samples")

Loaded 2764 training samples
Loaded 1373 test samples


In [4]:
# Preprocessing steps
# Choose a number of time steps
n_steps = 3
# Split into samples
X, y = split_sequence(humi_seq_train, n_steps)
# Reshape from [samples, timesteps] into [samples, timesteps, features]
n_features = 1
X = X.reshape((X.shape[0], X.shape[1], n_features))

In [25]:
# Define model
model = Sequential()
model.add(Conv1D(filters=64, kernel_size=2, activation='relu', input_shape=(n_steps, n_features)))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(50, activation='relu'))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

c:\Users\clong\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Fit model
model.fit(X, y, epochs=200, verbose=1)

In [27]:
# Predict on test set based on the steps
for i in range(10):
  random_num = random.randint(0, len(humi_seq_test)-4)
  x_input = np.array(humi_seq_test[random_num:random_num+n_steps])
  x_input = x_input.reshape((1, n_steps, n_features))
  predicted_value = model.predict(x_input, verbose=0)
  rmse = np.sqrt(np.mean((humi_seq_test[random_num+n_steps] - predicted_value[0][0])**2))
  print("Sequence:", np.array(humi_seq_test[random_num:random_num+n_steps]),"Next value:", humi_seq_test[random_num+n_steps], ", Predicted next value:", predicted_value[0][0], ", RMSE:", rmse)

Sequence: [32.7893 32.8827 33.0307] Next value: 33.112 , Predicted next value: 33.10311 , RMSE: 0.008888732910158126
Sequence: [43.1227 42.968  42.8053] Next value: 42.36 , Predicted next value: 42.864494 , RMSE: 0.5044943237304693
Sequence: [46.2667 46.016  45.6667] Next value: 45.4427 , Predicted next value: 45.792355 , RMSE: 0.3496545837402323
Sequence: [30.776  31.1107 31.4027] Next value: 31.6347 , Predicted next value: 31.567036 , RMSE: 0.06766432495117058
Sequence: [31.7573 31.9413 31.928 ] Next value: 31.9813 , Predicted next value: 31.97417 , RMSE: 0.007129315185547824
Sequence: [38.716  38.7747 38.7027] Next value: 38.6667 , Predicted next value: 38.73048 , RMSE: 0.06378019409179814
Sequence: [45.072  44.9387 44.8427] Next value: 44.8027 , Predicted next value: 44.8531 , RMSE: 0.05039982299804535
Sequence: [50.944  51.4587 51.9893] Next value: 52.3013 , Predicted next value: 52.254086 , RMSE: 0.047214459228513306
Sequence: [50.4933 50.3627 50.296 ] Next value: 50.2533 , Predi

## The multivariate model
This is an example to predict two different values (humidity & CO2) in 1 multivariate model

In [5]:
# Split a multivariate sequence into samples
def split_sequences(sequences, n_steps):
	X, y = list(), list()
	for i in range(len(sequences)):
		# Find the end of this pattern
		end_ix = i + n_steps
		# Check if we are beyond the dataset
		if end_ix > len(sequences)-1:
			break
		# Gather input and output parts of the pattern
		seq_x, seq_y = sequences[i:end_ix, :], sequences[end_ix, :]
		X.append(seq_x)
		y.append(seq_y)
	return np.array(X), np.array(y)

In [6]:
# Read given CO2 data in train and test sets
humi_seq_train = np.array(train_data['Relative_humidity_room'])
humi_seq_test = np.array(test_data['Relative_humidity_room'])
co2_seq_train = np.array(train_data['CO2_room'])
co2_seq_test = np.array(test_data['CO2_room'])

In [7]:
# Preprocessing steps
# Convert to [rows, columns] structure
humi_seq_train = humi_seq_train.reshape((len(humi_seq_train), 1))
co2_seq_train = co2_seq_train.reshape((len(co2_seq_train), 1))
# Horizontally stack columns
dataset = np.hstack((humi_seq_train, co2_seq_train))
# Choose a number of time steps
n_steps = 5
# Convert into input/output
X, y = split_sequences(dataset, n_steps)
# The dataset knows the number of features
n_features = X.shape[2]

In [8]:
# Define model
model2 = Sequential()
model2.add(Conv1D(filters=64, kernel_size=2, activation='relu', input_shape=(n_steps, n_features)))
model2.add(MaxPooling1D(pool_size=2))
model2.add(Flatten())
model2.add(Dense(50, activation='relu'))
model2.add(Dense(n_features))
model2.compile(optimizer='adam', loss='mse')

c:\Users\clong\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
# Fit model
model2.fit(X, y, epochs=200, verbose=1)

Epoch 1/200


87/87 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 4641.3589
Epoch 2/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 117.1718
Epoch 3/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 123.5587
Epoch 4/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 111.9466
Epoch 5/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 86.3268
Epoch 6/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 106.9288
Epoch 7/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 77.0172
Epoch 8/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 57.1099
Epoch 9/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 61.7539
Epoch 10/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 56.5781
Epoch 11/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 58.4353
Epoch 12/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 44.6162
Epoch 13/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 47.9662
Epoch 14/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 41.5489
Epoch 15/200
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 48.

In [ ]:
# # Predict on test set based on the steps
# for i in range(10):
#   random_num = random.randint(0, len(test_data)-4)
#   x_input = np.vstack((humi_seq_test[random_num:random_num+n_steps], co2_seq_test[random_num:random_num+n_steps])).T
#   x_input = x_input.reshape((1, n_steps, n_features))
#   predicted_value = model2.predict(x_input, verbose=0)
#   rmse_humi = np.sqrt(np.mean((humi_seq_test[random_num+n_steps] - predicted_value[0][0])**2))
#   rmse_co2 = np.sqrt(np.mean((co2_seq_test[random_num+n_steps] - predicted_value[0][1])**2))
#   print("Sequence", i)
#   print("Humidity sequence:", np.array(humi_seq_test[random_num:random_num+n_steps]), ", Next value:", humi_seq_test[random_num+n_steps], ", Predicted humidity:", predicted_value[0][0], ", RMSE:", rmse_humi)
#   print("CO2 sequence:", np.array(co2_seq_test[random_num:random_num+n_steps]), ", Next value:", co2_seq_test[random_num+n_steps], ", Predicted CO2:", predicted_value[0][1], ", RMSE:", rmse_co2)
#   print("-"*20)

In [11]:
import paho.mqtt.client as mqtt
import sys
import os
import random
import numpy as np
import json
import time

# MQTT Configuration
BROKER_ADDRESS = "app.coreiot.io"
PORT = 1883
CLIENT_ID = "iot_simulation_device"
ACCESS_TOKEN = "KT0MxDalCRkvSSDahLBX"  # Replace with your token

# Initialize MQTT client
client = mqtt.Client(CLIENT_ID)
client.username_pw_set(ACCESS_TOKEN)

# MQTT Callbacks
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("Connected to CoreIoT!")
    else:
        print(f"Failed to connect, return code {rc}")

def on_publish(client, userdata, mid):
    print(f"Message {mid} published successfully.")

client.on_connect = on_connect
client.on_publish = on_publish

# Reconnection logic
def ensure_connection():
    while not client.is_connected():
        try:
            # print("Attempting to reconnect...")
            # time.sleep(5)
            client.reconnect()
        except Exception as e:
            print(f"Reconnection failed: {e}")
            time.sleep(5)

# Connect to the MQTT broker
client.connect(BROKER_ADDRESS, PORT, 60)
client.loop_start()

# Simulate predictions and send telemetry data
for i in range(100):
    random_num = random.randint(0, len(test_data) - 4)
    x_input = np.vstack((humi_seq_test[random_num:random_num + n_steps], co2_seq_test[random_num:random_num + n_steps])).T
    x_input = x_input.reshape((1, n_steps, n_features))
    predicted_value = model2.predict(x_input, verbose=0)
    rmse_humi = np.sqrt(np.mean((humi_seq_test[random_num + n_steps] - predicted_value[0][0]) ** 2))
    rmse_co2 = np.sqrt(np.mean((co2_seq_test[random_num + n_steps] - predicted_value[0][1]) ** 2))
    
    # Real data
    real_humidity = humi_seq_test[random_num + n_steps]
    real_co2 = co2_seq_test[random_num + n_steps]
    
    # Print the results
    print("Sequence", i)
    print("Humidity sequence:", np.array(humi_seq_test[random_num:random_num + n_steps]), 
          ", Next value:", real_humidity, 
          ", Predicted humidity:", predicted_value[0][0], 
          ", RMSE:", rmse_humi)
    print("CO2 sequence:", np.array(co2_seq_test[random_num:random_num + n_steps]), 
          ", Next value:", real_co2, 
          ", Predicted CO2:", predicted_value[0][1], 
          ", RMSE:", rmse_co2)
    print("-" * 20)
    
    # Prepare telemetry data
    telemetry_data = {
        "predicted_humidity": float(predicted_value[0][0]),
        "predicted_co2": float(predicted_value[0][1]),
        "real_humidity": float(real_humidity),
        "real_co2": float(real_co2),
        "rmse_humidity": float(rmse_humi),
        "rmse_co2": float(rmse_co2)
    }
    
    # Ensure the client is connected before publishing
    ensure_connection()
    try:
        result = client.publish("v1/devices/me/telemetry", json.dumps(telemetry_data), qos=1)
        result.wait_for_publish()  # Ensure the message is sent
        print(f"Sent telemetry: {telemetry_data}")
    except RuntimeError as e:
        print(f"Failed to publish message: {e}. Retrying...")
        ensure_connection()  # Reconnect and retry
        continue  # Skip to the next iteration

    time.sleep(1)  # Sleep for 1 second between messages

# Stop the MQTT client loop
client.loop_stop()
client.disconnect()

Connected to CoreIoT!
Sequence 0
Humidity sequence: [36.9653 37.0373 37.8347 38.0853 38.472 ] , Next value: 38.56 , Predicted humidity: 38.097305 , RMSE: 0.4626947021484398
CO2 sequence: [209.44  208.981 209.632 209.728 209.312] , Next value: 210.261 , Predicted CO2: 212.035 , RMSE: 1.7740036621093793
--------------------
Message 1 published successfully.
Sent telemetry: {'predicted_humidity': 38.09730529785156, 'predicted_co2': 212.03500366210938, 'real_humidity': 38.56, 'real_co2': 210.261, 'rmse_humidity': 0.4626947021484398, 'rmse_co2': 1.7740036621093793}
Sequence 1
Humidity sequence: [45.032  44.9653 44.8267 44.744  44.7947] , Next value: 44.912 , Predicted humidity: 44.797737 , RMSE: 0.11426287841796778
CO2 sequence: [197.589 196.821 196.789 197.44  196.8  ] , Next value: 196.395 , Predicted CO2: 199.91539 , RMSE: 3.5203900146484273
--------------------
Message 2 published successfully.
Sent telemetry: {'predicted_humidity': 44.79773712158203, 'predicted_co2': 199.91539001464844

0